In [1]:
import sys

import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

3.12.6 2.0.2 2.3.3 1.6.1


   SEED = 421 (last 4 digits of my student ID: 0421)

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 421 

data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c != "disease"]
X, y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

(297, 14) 0.461 (207, 13) (90, 13)


In [3]:
toy = pd.DataFrame({
    "age_55_plus": [1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
    "exercise_angina": [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "male": [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    "high_bp": [1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0],
    "disease": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})
toy

,age_55_plus,exercise_angina,male,high_bp,disease
0,1,1,1,1,1
1,1,1,1,1,1
2,1,1,1,0,1
3,1,1,0,1,1
4,1,0,0,1,1
5,0,0,0,0,1
6,1,0,1,0,0
7,0,0,1,0,0
8,0,0,1,0,0
9,0,0,0,1,0


H = -(6/12 · log2(6/12) + 6/12 · log2(6/12)) = -(0.5·(-1) + 0.5·(-1)) = 1

In [4]:
import math
H_parent = -(0.5 * math.log2(0.5) + 0.5 * math.log2(0.5))
print(H_parent)

1.0


In [5]:
def H(*counts):
    n = sum(counts)
    return -sum((c / n) * math.log2(c / n) for c in counts if c > 0)

print(pd.crosstab(toy["age_55_plus"], toy["disease"]))
gain_age = 1 - (6/12 * H(5, 1) + 6/12 * H(1, 5))
print(gain_age)

disease      0  1
age_55_plus      
0            5  1
1            1  5
0.3499775783516459


import math

def H(*counts):
    """Entropy in bits of a group given class counts."""
    n = sum(counts)
    return -sum((c / n) * math.log2(c / n) for c in counts if c > 0)

H_parent = H(6, 6)
print("H_parent =", H_parent)
for f in ["age_55_plus", "exercise_angina", "male", "high_bp"]:
    print(f)
    print(pd.crosstab(toy[f], toy["disease"]), "\n")
gain_age = H_parent - (6/12 * H(5, 1) + 6/12 * H(1, 5))
gain_angina = H_parent - (8/12 * H(6, 2) + 4/12 * H(0, 4))
gain_male = H_parent - (6/12 * H(3, 3) + 6/12 * H(3, 3))
gain_bp = H_parent - (6/12 * H(4, 2) + 6/12 * H(2, 4))

print("gain age_55_plus    =", round(gain_age, 4))
print("gain exercise_angina =", round(gain_angina, 4))
print("gain male            =", round(gain_male, 4))
print("gain high_bp         =", round(gain_bp, 4))

In [8]:
sub = toy[toy["exercise_angina"] == 0]
print(sub, "\n")

H_sub = H(6, 2)  
print("H_sub =", round(H_sub, 4))

for f in ["age_55_plus", "male", "high_bp"]:
    print(f)
    print(pd.crosstab(sub[f], sub["disease"]), "\n")
gain_age_sub = H_sub - (6/8 * H(5, 1) + 2/8 * H(1, 1))
gain_male_sub = H_sub - (5/8 * H(3, 2) + 3/8 * H(3, 0))
gain_bp_sub = H_sub - (5/8 * H(4, 1) + 3/8 * H(2, 1))

print("gain age_55_plus =", round(gain_age_sub, 4))
print("gain male        =", round(gain_male_sub, 4))
print("gain high_bp     =", round(gain_bp_sub, 4))

    age_55_plus  exercise_angina  male  high_bp  disease
4             1                0     0        1        1
5             0                0     0        0        1
6             1                0     1        0        0
7             0                0     1        0        0
8             0                0     1        0        0
9             0                0     0        1        0
10            0                0     0        1        0
11            0                0     0        0        0 

H_sub = 0.8113
age_55_plus
disease      0  1
age_55_plus      
0            5  1
1            1  1 

male
disease  0  1
male         
0        3  2
1        3  0 

high_bp
disease  0  1
high_bp      
0        4  1
1        2  1 

gain age_55_plus = 0.0738
gain male        = 0.2044
gain high_bp     = 0.0157


# Part 2: Decision tree from scratch

In [9]:
def entropy(y):
    """Entropy in bits of an array of integer class labels."""
    y = np.asarray(y)
    if len(y) == 0:
        return 0.0
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))


def information_gain(y, left):
    """Entropy of y minus the size-weighted entropy of y[left] and y[~left]; left is a boolean mask."""
    n = len(y)
    n_left = int(left.sum())
    n_right = n - n_left
    return entropy(y) - (n_left / n) * entropy(y[left]) - (n_right / n) * entropy(y[~left])


def best_split(X, y, min_samples_leaf=1):
    """Return (feature_index, threshold, gain) of the best split, or None if no valid split exists."""
    best = None
    for j in range(X.shape[1]):
        values = np.unique(X[:, j])
        for t in (values[:-1] + values[1:]) / 2:      # midpoints between consecutive distinct values
            left = X[:, j] <= t
            n_left = int(left.sum())
            if n_left < min_samples_leaf or len(y) - n_left < min_samples_leaf:
                continue
            g = information_gain(y, left)
            if best is None or g > best[2]:           # strict ">" keeps lower feature / lower threshold on ties
                best = (j, float(t), g)
    return best


def majority(y):
    """Majority class; a tie goes to the smaller label."""
    return int(np.bincount(y).argmax())


def build_tree(X, y, max_depth=None, min_samples_leaf=1, depth=0):
    """Grow a tree recursively and return it as nested dicts (or any structure you like)."""
    if len(np.unique(y)) == 1 or (max_depth is not None and depth == max_depth):
        return {"leaf": majority(y)}
    split = best_split(X, y, min_samples_leaf)
    if split is None:
        return {"leaf": majority(y)}
    j, t, _ = split
    left = X[:, j] <= t
    return {"feature": j, "threshold": t,
            "left": build_tree(X[left], y[left], max_depth, min_samples_leaf, depth + 1),
            "right": build_tree(X[~left], y[~left], max_depth, min_samples_leaf, depth + 1)}


def predict_tree(tree, X):
    """Return one predicted label per row of X."""
    out = np.empty(len(X), dtype=int)
    for i, row in enumerate(X):
        node = tree
        while "leaf" not in node:
            node = node["left"] if row[node["feature"]] <= node["threshold"] else node["right"]
        out[i] = node["leaf"]
    return out

In [10]:
y_toy = toy["disease"].to_numpy()
print("entropy:", entropy(y_toy))   # ожидается 1.0
for f in ["age_55_plus", "exercise_angina", "male", "high_bp"]:
    left = toy[f].to_numpy() == 0
    print(f, round(information_gain(y_toy, left), 4))

entropy: 1.0
age_55_plus 0.35
exercise_angina 0.4591
male 0.0
high_bp 0.0817


In [11]:
from sklearn.tree import DecisionTreeClassifier

ref = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=SEED).fit(X_train, y_train)
print("sklearn:", feature_names[ref.tree_.feature[0]], ref.tree_.threshold[0])

j, t, g = best_split(X_train, y_train)
print("mine:   ", feature_names[j], t, "gain =", round(g, 4))

sklearn: thal 4.5
mine:    thal 4.5 gain = 0.1897


In [12]:
rows = []
for d in [1, 2, 3, 4, 5, None]:
    mine = build_tree(X_train, y_train, max_depth=d)
    sk = DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=SEED).fit(X_train, y_train)
    agree = (predict_tree(mine, X_test) == sk.predict(X_test)).mean()
    rows.append({"max_depth": "unlimited" if d is None else d, "agreement": round(agree, 4)})
pd.DataFrame(rows)

,max_depth,agreement
0,1,1.0000
1,2,1.0000
2,3,1.0000
3,4,1.0000
4,5,1.0000
5,unlimited,0.9667
